#### exequting the skip gram model and its three variations

-  negative sampeling

-  contrastive estimation 

-  hirerchial softmax

In [2]:
import numpy as np
import nltk
import matplotlib.pyplot as plt
from IPython.display import clear_output

# from scipy.sparse.linalg import svds
# import sys

nltk.download('brown')

from nltk.corpus import brown


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\sp710\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [3]:
# constants

WINDOW_SIZE: int=2

k:int= 10

In [4]:
sentences=brown.sents()[:100]

corpus = [sentence[i:i+WINDOW_SIZE] for sentence in sentences for i in range(len(sentence)-WINDOW_SIZE+1)]

print(corpus)


words=sorted(set(word for sentence in sentences for word in sentence))

word2id = {w: i for i, w in enumerate(words)}
id2word = {i: w for w, i in word2id.items()}


[['The', 'Fulton'], ['Fulton', 'County'], ['County', 'Grand'], ['Grand', 'Jury'], ['Jury', 'said'], ['said', 'Friday'], ['Friday', 'an'], ['an', 'investigation'], ['investigation', 'of'], ['of', "Atlanta's"], ["Atlanta's", 'recent'], ['recent', 'primary'], ['primary', 'election'], ['election', 'produced'], ['produced', '``'], ['``', 'no'], ['no', 'evidence'], ['evidence', "''"], ["''", 'that'], ['that', 'any'], ['any', 'irregularities'], ['irregularities', 'took'], ['took', 'place'], ['place', '.'], ['The', 'jury'], ['jury', 'further'], ['further', 'said'], ['said', 'in'], ['in', 'term-end'], ['term-end', 'presentments'], ['presentments', 'that'], ['that', 'the'], ['the', 'City'], ['City', 'Executive'], ['Executive', 'Committee'], ['Committee', ','], [',', 'which'], ['which', 'had'], ['had', 'over-all'], ['over-all', 'charge'], ['charge', 'of'], ['of', 'the'], ['the', 'election'], ['election', ','], [',', '``'], ['``', 'deserves'], ['deserves', 'the'], ['the', 'praise'], ['praise', 'an

In [15]:
# constants

L=2 #is the no if layer

MAX_ITR=100 

LEARNING_RATE=0.2 #η 

NO_OF_INPUT=[len(words),k,len(words)] # mem friendly input h1,h2 output

NO_OF_OUTPUT=len(words)

BATCH_SIZE=40

gamma=0.1 # trust on the previous gradient


In [6]:

def initialize_parameters(layer_dims, method="xavier", select_bias: bool=True):
    """
    Initializes weights and biases for a fully connected neural network.
    
    Parameters:
    -----------
    layer_dims : list of int
        Sizes of each layer in the network. Example: [784, 128, 64, 10]
    method : str
        Initialization method: "xavier" or "he"
    
    Returns:
    --------
    weights : list of np.ndarray
        Weight matrices for each layer
    biases : list of np.ndarray
        Bias vectors for each layer
    """
    weights = [] #(784,128) , (128,64) ,(64,10)
    biases = [] #(764 x 1 , 128 x 1 , 64 x 1 , 10 x 1)
    
    for i in range(len(layer_dims)-1):
        n_in = layer_dims[i]
        n_out = layer_dims[i+1]
        
        if method == "xavier":
            W = np.random.randn(n_in, n_out) * np.sqrt(1.0 / n_in)
        elif method == "he":
            W = np.random.randn(n_in,n_out) * np.sqrt(2.0 / n_in)
        else:
            raise ValueError("Invalid method. Use 'xavier' or 'he'.")
        
        if select_bias : b = np.zeros((1, n_out))
        
        weights.append(W)
        
        if select_bias : biases.append(b)
    
    return weights, biases

weight , _ =initialize_parameters(NO_OF_INPUT,select_bias=False)

print(weight[0].shape)
print(weight[1].shape)



(860, 10)
(10, 860)


In [8]:
# helper functionns

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def loss_fn(y_pred, y_true):
    """
    Cross-entropy loss
    y_pred: (batch_size, vocab_size)
    y_true: (batch_size, vocab_size) one-hot
    """
    m = y_true.shape[0]
    return -np.sum(y_true * np.log(y_pred + 1e-9)) / m

def eigen(output, word2id, vocab_size):
    """
    Convert words to one-hot vectors
    output: list/array of words (batch_size,)
    """
    idxs = [word2id[w] for w in output]
    ans = np.eye(vocab_size)[idxs]
    return ans

def update_parameters(grads_w, grads_b):
    for i in range(L):
        weight[i] -= LEARNING_RATE * grads_w[i] 
        # bias[i]   -= LEARNING_RATE * grads_b[i]

# to modify
def forwardpropogation(x , ): 
    """
    Forward pass
    x: (batch_size, input_dim)
    """
    activation, preactivation = [x], []

    for i in range(L):
        z = np.dot(activation[-1], weight[i]) 
        preactivation.append(z)

        if i == L - 1:
            a = softmax(z)
        else :
            a=z
        activation.append(a)

    return activation, preactivation

## to modify
def backpropogation(activation, preactivation, y):
    """
    Backward pass
    y: one-hot (batch_size, vocab_size)
    """
    m = y.shape[0]
    grad_w = [None] * L
    grad_b = [None] * L

    # output layer error
    dz = activation[-1] - y  

    for i in reversed(range(L)):
        A_prev = activation[i]

        grad_w[i] = np.dot(A_prev.T, dz) / m
        # grad_b[i] = np.sum(dz, axis=0, keepdims=True) / m

        if i > 0:
            dA_prev = np.dot(dz, weight[i].T)
            dz = dA_prev

    return grad_w, grad_b

def windows_to_concat_onehot(batch_windows, word2id, vocab_size):
    """
    batch_windows: (batch_size, WINDOW_SIZE-1) of words
    returns: (batch_size, (WINDOW_SIZE-1)*vocab_size) concatenated one-hot
    """
    batch_size, context_len = batch_windows.shape
    idxs = np.vectorize(word2id.get)(batch_windows)   # convert words -> indices
    onehots = np.eye(vocab_size)[idxs]                # (batch, context_len, vocab_size)
    return onehots.reshape(batch_size, -1)      


In [9]:

np_corpus = np.array(corpus)
print("Corpus shape:", np_corpus.shape)

split_idx = int(0.7 * np_corpus.shape[0])
X_train = np_corpus[:split_idx, 0:1]          # center words
Y_train = np_corpus[:split_idx, 1:WINDOW_SIZE]  # context words

X_test  = np_corpus[split_idx:, 0:1]          # center words
Y_test = np_corpus[split_idx:, 1:WINDOW_SIZE]  # context words


vocab_size = len(word2id)

X_idx = np.vectorize(word2id.get)(X_train.flatten())  

print(X_idx)
X_train = np.eye(vocab_size)[X_idx]              


Y_idx = np.vectorize(word2id.get)(Y_train)            
Y_train = np.eye(vocab_size)[Y_idx]             

Y_train = Y_train.reshape(Y_train.shape[0], -1)  

print("X_train_1hot shape:", X_train.shape)
print("Y_train_1hot shape:", Y_train.shape)
print("Y_train_1hot_concat shape:", Y_train.shape)


Corpus shape: (2168, 2)
[188  96  75 ... 481   9 780]
X_train_1hot shape: (1517, 860)
Y_train_1hot shape: (1517, 860)
Y_train_1hot_concat shape: (1517, 860)


In [18]:
def update_parameters_wt_movmentum(grads_w,grads_b,history):
    for i in range(L):
        term_w=(LEARNING_RATE * grads_w[i] + 0.5*(history[i][0]))
        # term_b=(LEARNING_RATE * grads_b[i] + 0.5*(history[i][1]))
        weight[i] -= term_w
        # bias[i] -= term_b
        history[i] = (term_w)

def update_parameters_nag(grads_w, grads_b, velocity):
    global weight
    for i in range(L):
        v_prev = velocity[i].copy()
        velocity[i] = gamma * velocity[i] + LEARNING_RATE * grads_w[i]
        weight[i] -= -gamma * v_prev + (1 + gamma) * velocity[i]
    return velocity

history=[]


for i in range(L):
    history.append(np.zeros_like(weight[i]))

for epoch in range(MAX_ITR):
    
    idx = np.arange(X_train.shape[0])
    np.random.shuffle(idx)
    X_train = X_train[idx]
    Y_train = Y_train[idx]

    print(f"\nEpoch {epoch+1}/{MAX_ITR}")
    print("Sample batch:", X_train[:2], Y_train[:2])

    for i in range(0, len(X_train), BATCH_SIZE):
        
        X_batch = X_train[i : i + BATCH_SIZE]               
        Y_batch = Y_train[i : i + BATCH_SIZE]   

        activation, preactivation = forwardpropogation(X_batch)

        loss = loss_fn(activation[-1], Y_batch)
        
        if i % (BATCH_SIZE * 10) == 0:
            print(f"  batch {i//BATCH_SIZE:03d}: loss={loss:.4f}")


        dW, db = backpropogation(activation, preactivation, Y_batch)
        # update_parameters(dW, db)
        update_parameters_nag(dW,db,history)



Epoch 1/100
Sample batch: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]] [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
  batch 000: loss=1.4958
  batch 010: loss=1.1243
  batch 020: loss=1.5178
  batch 030: loss=1.6101

Epoch 2/100
Sample batch: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]] [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
  batch 000: loss=1.2430
  batch 010: loss=1.0514
  batch 020: loss=1.8379
  batch 030: loss=1.8407

Epoch 3/100
Sample batch: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]] [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
  batch 000: loss=1.1813
  batch 010: loss=1.7802
  batch 020: loss=1.5072
  batch 030: loss=1.5775

Epoch 4/100
Sample batch: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]] [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
  batch 000: loss=1.4165
  batch 010: loss=1.1529
  batch 020: loss=1.2826
  batch 030: loss=1.4773

Epoch 5/100
Sample batch: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]] [[0. 0